# **Feature Engineering & NLP**
**Project**: Airbnb Market Segmentation Analysis \
**Goal**: Build the clustering-ready feature set from `listings_clean` (structural + domain-driven features, encoded for K-Prototypes), then extend it with NLP-derived signals from listing text & reviews text data.

## 1. Imports & environment setup

In [1]:
import pandas as pd
import numpy as np
import re
import html

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
# Base path configuration
BASE_PATH = Path.cwd().parent

## 2. Data Loading

In [3]:
listings_clean = pd.read_csv(BASE_PATH / 'data' / 'processed' / 'listings_clean.csv')
reviews_clean = pd.read_csv(BASE_PATH / 'data' / 'processed' / 'reviews_clean.csv')

#### 2.1 Listings dataframe

In [4]:
listings_clean.shape

(35818, 72)

In [5]:
listings_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 35818 entries, 0 to 35817
Data columns (total 72 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   id                                  35818 non-null  int64  
 1   host_id                             35818 non-null  int64  
 2   host_response_time                  35818 non-null  str    
 3   host_response_rate_raw              29444 non-null  float64
 4   host_acceptance_rate_raw            32319 non-null  float64
 5   host_is_superhost                   35818 non-null  int64  
 6   host_neighbourhood                  35818 non-null  str    
 7   host_listings_count                 35818 non-null  float64
 8   host_total_listings_count           35818 non-null  float64
 9   host_has_profile_pic                35818 non-null  int64  
 10  host_identity_verified              35818 non-null  int64  
 11  neighbourhood_cleansed              35818 non-null  

In [6]:
listings_clean.describe(include = 'all')

,id,host_id,host_response_time,host_response_rate_raw,host_acceptance_rate_raw,host_is_superhost,host_neighbourhood,host_listings_count,host_total_listings_count,host_has_profile_pic,...,review_scores_accuracy_binned,review_scores_cleanliness_binned,review_scores_checkin_binned,review_scores_communication_binned,review_scores_location_binned,review_scores_value_binned,host_acceptance_rate_binned,host_response_rate_binned,reviews_per_month_binned,last_review_recency_binned
count,3.581800e+04,3.581800e+04,35818,29444.000000,32319.000000,35818.000000,35818,35818.000000,35818.000000,35818.000000,...,35818,35818,35818,35818,35818,35818,35818,35818,35818,35818
unique,NaN,NaN,5,NaN,NaN,NaN,100,NaN,NaN,NaN,...,3,3,3,3,3,3,3,3,4,4
top,NaN,NaN,within an hour,NaN,NaN,NaN,Unknown,NaN,NaN,NaN,...,Above Average,Above Average,Above Average,Above Average,Above Average,Below Average,Full Acceptance,Full Response,Moderate Demand,Active
freq,NaN,NaN,25098,NaN,NaN,NaN,24751,NaN,NaN,NaN,...,21507,20021,24267,24935,17700,16983,21295,24628,14132,20557
mean,7.541191e+17,2.666308e+08,NaN,0.966612,0.926700,0.399213,NaN,9.286867,11.760902,0.939835,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,5.746268e+17,2.439774e+08,NaN,0.136872,0.201959,0.489743,NaN,20.757329,29.960408,0.237796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.183400e+04,2.353000e+03,NaN,0.000000,0.000000,0.000000,NaN,1.000000,1.000000,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,4.020579e+07,3.129867e+07,NaN,1.000000,0.980000,0.000000,NaN,1.000000,1.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,9.565242e+17,1.901475e+08,NaN,1.000000,1.000000,0.000000,NaN,3.000000,3.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.271719e+18,5.070057e+08,NaN,1.000000,1.000000,1.000000,NaN,7.000000,8.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### 2.2 Reviews dataframe

In [7]:
reviews_clean.shape

(2193159, 5)

In [8]:
reviews_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 2193159 entries, 0 to 2193158
Data columns (total 5 columns):
 #   Column          Dtype
---  ------          -----
 0   listing_id      int64
 1   date            str  
 2   reviewer_id     int64
 3   comments_clean  str  
 4   review_length   int64
dtypes: int64(3), str(2)
memory usage: 83.7 MB


In [9]:
reviews_clean.describe(include = 'all')

,listing_id,date,reviewer_id,comments_clean,review_length
count,2.193159e+06,2193159,2.193159e+06,2193122,2.193159e+06
unique,NaN,5242,NaN,2123477,NaN
top,NaN,2025-06-15,NaN,unknown,NaN
freq,NaN,3059,NaN,4095,NaN
mean,2.786878e+17,NaN,1.988891e+08,NaN,2.691474e+02
std,4.496488e+17,NaN,1.838524e+08,NaN,2.415132e+02
min,2.737000e+03,NaN,4.600000e+01,NaN,1.000000e+00
25%,9.950970e+06,NaN,4.574545e+07,NaN,1.030000e+02
50%,2.920817e+07,NaN,1.344770e+08,NaN,2.050000e+02
75%,6.590309e+17,NaN,3.235598e+08,NaN,3.600000e+02
